# Chapter 2 — Integrating an orbit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/02_integrating_orbits.ipynb)

**Goal.** Write an orbit integrator from scratch, and learn how to tell whether its output
can be trusted.

**You will learn**
1. why the simplest method (Euler) fails for orbits;
2. the **leapfrog** integrator, the workhorse of galactic dynamics;
3. the difference between *accuracy* (how close the orbit is to the true one) and
   *conservation* (whether energy drifts over many orbits);
4. how to measure an integrator's order of convergence against an exact solution;
5. how AGAMA's orbit integrator compares — and how to test it the same way.

**Units.** $G = 1$ ($G$ the gravitational constant), as in Chapter 1. We use the Kepler problem (a point of mass $M = 1$) because its
orbits are known exactly, and the Plummer sphere (mass $M = 1$, scale radius $b = 1$) from Chapter 1.

*In AGAMA* all of this is done by `agama.orbit` (Chapter 0). Writing our own integrator first shows
what such a function does and what can go wrong; Section 2.7 then puts AGAMA through the same tests.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp   # SciPy's general-purpose ODE solver (used in §2.5)
from scipy.optimize import brentq       # root finding (used to solve Kepler's equation)

G, M, b = 1.0, 1.0, 1.0

## 2.1 Equations of motion

A star's state is its position $\mathbf x$ and velocity $\mathbf v$. Newton's second law
gives two first-order equations:

$$ \frac{d\mathbf x}{dt} = \mathbf v, \qquad \frac{d\mathbf v}{dt} = \mathbf g(\mathbf x) = -\nabla\Phi , $$

where $t$ is time, $\Phi$ the gravitational potential, $\mathbf g$ the gravitational acceleration (force per unit mass), and
$\nabla\Phi$ the gradient of $\Phi$, the vector of its derivatives along the Cartesian axes $x, y, z$; we write
$\mathbf x = (x, y, z)$ and $\mathbf v = (v_x, v_y, v_z)$. The orbits in this chapter stay in the $x$–$y$ plane.

Orbits in a spherical potential stay in a plane (Chapter 1), so two dimensions suffice:
positions and velocities are numpy arrays of 2 numbers. The accelerations of a point mass,
$\mathbf g = -GM\mathbf x/r^3$ with $r = |\mathbf x|$, and of a Plummer sphere, $\mathbf g = -GM\mathbf x/(r^2+b^2)^{3/2}$, are

In [ ]:
def accel_kepler(x):
    r = np.linalg.norm(x)                        # |x| = sqrt(x^2 + y^2)
    return -G * M * x / r**3

def accel_plummer(x):
    return -G * M * x / (np.sum(x**2) + b**2)**1.5

def energy(x, v, phi):
    """Energy per unit mass, E = v^2/2 + Phi(r).

    Works for a single state (x, v of shape (2,)) or a whole trajectory (shape (n, 2)):
    axis=-1 means "sum over the last index", i.e. over the x and y components.
    """
    return 0.5 * np.sum(v**2, axis=-1) + phi(np.linalg.norm(x, axis=-1))

phi_kepler = lambda r: -G * M / r
phi_plummer = lambda r: -G * M / np.sqrt(r**2 + b**2)

As a test orbit we take a Kepler ellipse with semi-major axis $a = 1$ and eccentricity
$e = 0.5$, started at pericentre $r_p = a(1-e)$. Its period is $P = 2\pi\sqrt{a^3/GM} = 2\pi$
(Kepler's third law), and its speed at pericentre follows from the *vis-viva* equation
$v^2 = GM(2/r - 1/a)$ for the speed $v$ at distance $r$, which at $r = a(1-e)$ gives the speed at pericentre, $v_p$: $v_p^2 = \frac{GM}{a}\frac{1+e}{1-e}$.

In [ ]:
a, e = 1.0, 0.5
P = 2 * np.pi * np.sqrt(a**3 / (G * M))
x0 = np.array([a * (1 - e), 0.0])                               # pericentre on the x-axis
v0 = np.array([0.0, np.sqrt(G * M / a * (1 + e) / (1 - e))])    # velocity perpendicular to x
E0 = energy(x0, v0, phi_kepler)
print(f'period P = {P:.6f},  energy E = {E0:.6f}  (expected -GM/2a = {-G * M / (2 * a)})')

## 2.2 Euler's method — and why it fails

The most obvious scheme advances both variables using their current rates of change, over a
small time step $h$:

$$ \mathbf x_{n+1} = \mathbf x_n + h\,\mathbf v_n, \qquad \mathbf v_{n+1} = \mathbf v_n + h\,\mathbf g(\mathbf x_n), $$

where $\mathbf x_n$ and $\mathbf v_n$ are the position and velocity after $n$ steps, at time $t_n = n h$.

In [ ]:
def euler(accel, x0, v0, h, nsteps):
    """Euler integration; returns positions and velocities at all steps, each of shape (nsteps+1, 2)."""
    x, v = np.empty((nsteps + 1, 2)), np.empty((nsteps + 1, 2))   # pre-allocate the output arrays
    x[0], v[0] = x0, v0
    for n in range(nsteps):
        x[n + 1] = x[n] + h * v[n]
        v[n + 1] = v[n] + h * accel(x[n])
    return x, v

h = P / 1000                                                  # 1000 steps per orbit
x_eu, v_eu = euler(accel_kepler, x0, v0, h, 10 * 1000)        # 10 orbits

plt.figure(figsize=(5, 5))
plt.plot(x_eu[:, 0], x_eu[:, 1], lw=0.7)                      # column 0 = x, column 1 = y
plt.plot(0, 0, 'k*', ms=12)                                   # the central mass
plt.gca().set_aspect('equal')                                 # same scale on both axes, so ellipses look right
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.title('Euler, 1000 steps per orbit, 10 orbits'); plt.show()
print(f'energy after 10 orbits: {energy(x_eu[-1], v_eu[-1], phi_kepler):.4f}  (started at {E0:.4f})')

**Figure 2.1.** A Kepler orbit ($a = 1$, $e = 0.5$) integrated with Euler's method for 10 orbits at 1000 steps per orbit, in the $x$–$y$ plane. The star marks the central mass.

The orbit should be a closed ellipse; instead it spirals outwards (Figure 2.1). Euler's method adds a little
energy at every step, and the errors accumulate. Using smaller steps only delays the problem.
(Why outwards? Euler moves the star along the *current* velocity, i.e. along the tangent to the
orbit, and a tangent line always leads slightly outside a curve that bends towards the centre.)

## 2.3 The leapfrog integrator

Leapfrog ("kick–drift–kick") updates position and velocity in a staggered way:

$$
\begin{aligned}
\mathbf v_{n+1/2} &= \mathbf v_n + \tfrac h2\, \mathbf g(\mathbf x_n) && \text{(half kick)}\\
\mathbf x_{n+1}   &= \mathbf x_n + h\, \mathbf v_{n+1/2} && \text{(drift)}\\
\mathbf v_{n+1}   &= \mathbf v_{n+1/2} + \tfrac h2\, \mathbf g(\mathbf x_{n+1}) && \text{(half kick)}
\end{aligned}
$$

Here $\mathbf v_{n+1/2}$ is the velocity at the half step, $t_n + h/2$. The position is advanced with the velocity at the *middle* of the step, which is what makes it
more accurate than Euler. It costs one force evaluation per step, the same as Euler (the force at
the end of one step is reused at the start of the next). It has two properties that Euler lacks:
- it is **time-reversible**: running it backwards with $-h$ retraces the path exactly;
- it is **symplectic**: it preserves the geometry of phase space, and as a result nearly conserves
  a slightly perturbed energy (over exponentially long times), so the energy error oscillates but does not grow.

In [ ]:
def leapfrog(accel, x0, v0, h, nsteps):
    """Kick-drift-kick leapfrog. Returns positions and velocities, shape (nsteps+1, dim)."""
    x, v = np.empty((nsteps + 1, len(x0))), np.empty((nsteps + 1, len(x0)))
    x[0], v[0] = x0, v0
    g = accel(x0)                          # force at the start
    for n in range(nsteps):
        v_half = v[n] + 0.5 * h * g        # half kick
        x[n + 1] = x[n] + h * v_half       # drift
        g = accel(x[n + 1])                # one new force evaluation per step ...
        v[n + 1] = v_half + 0.5 * h * g    # ... reused for this half kick and the next one
    return x, v

x_lf, v_lf = leapfrog(accel_kepler, x0, v0, h, 10 * 1000)
t = np.arange(len(x_lf)) * h               # time of each stored step

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(x_lf[:, 0], x_lf[:, 1], lw=0.7); ax[0].plot(0, 0, 'k*', ms=12)
ax[0].set_aspect('equal'); ax[0].set_xlabel('$x$'); ax[0].set_ylabel('$y$')
ax[0].set_title('leapfrog, 1000 steps per orbit, 10 orbits')
# relative energy error |E/E0 - 1| on a log scale, for both methods
ax[1].semilogy(t / P, np.abs(energy(x_eu, v_eu, phi_kepler) / E0 - 1), label='Euler')
ax[1].semilogy(t / P, np.abs(energy(x_lf, v_lf, phi_kepler) / E0 - 1), label='leapfrog')
ax[1].set_xlabel('time / orbital period'); ax[1].set_ylabel(r'$|\Delta E / E|$'); ax[1].legend()
plt.tight_layout(); plt.show()

**Figure 2.2.** Left: the same orbit integrated with leapfrog. Right: relative energy error $|\Delta E/E|$ ($\Delta E$ the change of the energy $E$ since the start) against time in orbital periods, for Euler and leapfrog.

The leapfrog orbit closes on itself (Figure 2.2, left); its energy error (right) is largest near pericentre (where the
star moves fastest and the force changes most within one step) and returns to small values every orbit.

**Time reversibility.** Integrate forward, flip the velocity, integrate the same number of
steps again: we should land back at the starting point, up to round-off (about $10^{-16}$
per operation in double precision).

In [ ]:
x_f, v_f = leapfrog(accel_kepler, x0, v0, h, 5000)              # forward 5000 steps
x_b, v_b = leapfrog(accel_kepler, x_f[-1], -v_f[-1], h, 5000)   # from the end, with reversed velocity
err_rev = np.linalg.norm(x_b[-1] - x0)                          # [-1] is the last element
print(f'distance from start after going forward and back 5000 steps: {err_rev:.1e}')
assert err_rev < 1e-10

## 2.4 How accurate is it? Convergence against the exact solution

The Kepler orbit is known exactly at any time. With mean motion $n = \sqrt{GM/a^3}$ (the average
angular speed; not the step index $n$ of §2.2), solve **Kepler's equation** $\eta - e\sin\eta = n t$ for the *eccentric anomaly*
$\eta$; then

$$ x = a(\cos\eta - e), \qquad y = a\sqrt{1-e^2}\,\sin\eta , $$

the position in the plane of the orbit, with the central mass at the origin, the pericentre on the $+x$ axis and
time $t$ counted from a pericentre passage.

Kepler's equation has no closed-form solution, so we solve it numerically with `brentq`. Since
$|e\sin\eta| \le e < 1$, the root lies within 1 of $nt$, which gives the bracketing interval.

In [ ]:
def kepler_position(t, a=a, e=e):
    """Exact position on the Kepler ellipse at time t, starting from pericentre at t = 0."""
    mean_anomaly = np.sqrt(G * M / a**3) * t
    eta = brentq(lambda s: s - e * np.sin(s) - mean_anomaly, mean_anomaly - 1, mean_anomaly + 1, xtol=1e-15)
    return np.array([a * (np.cos(eta) - e), a * np.sqrt(1 - e**2) * np.sin(eta)])

# compare the leapfrog orbit from §2.3 with the exact one; t[::100] takes every 100th time,
# i.e. 10 samples per orbit
x_exact = np.array([kepler_position(ti) for ti in t[::100]])
dx = np.linalg.norm(x_lf[::100] - x_exact, axis=1)
print(f'position error with h = P/1000: after 1 orbit {dx[10]:.1e}, after 10 orbits {dx[-1]:.1e}')

The energy error is bounded, but the position error grows: the integrated star runs slightly
ahead of or behind the true one, and this *phase error* accumulates roughly linearly with time.
Conserving energy is not the same as being accurate.

Now measure the position error at a fixed time $t_{\rm end} = 2.3P$ (not a whole number of
orbits, so that errors cannot cancel by symmetry) while halving the step size. For a method of
order $p$ the error scales as $h^p$: halving $h$ should divide the error by $2^p$. On a log–log
plot of error against $h$, a method of order $p$ is a straight line of slope $p$.

In [ ]:
t_end = 2.3 * P
x_true = kepler_position(t_end)
steps_per_orbit = np.array([250, 500, 1000, 2000, 4000, 8000])   # each twice the previous
err_lf, err_eu = [], []
for N in steps_per_orbit:
    nsteps = int(round(t_end / P * N))
    hN = t_end / nsteps                                   # exact step so that we end at t_end
    # leapfrog(...) returns (x, v); [0] takes x, and [-1] its final row
    err_lf.append(np.linalg.norm(leapfrog(accel_kepler, x0, v0, hN, nsteps)[0][-1] - x_true))
    err_eu.append(np.linalg.norm(euler(accel_kepler, x0, v0, hN, nsteps)[0][-1] - x_true))
err_lf, err_eu = np.array(err_lf), np.array(err_eu)
hs = P / steps_per_orbit

plt.figure(figsize=(6, 4.5))
plt.loglog(hs, err_eu, 'o-', label='Euler')
plt.loglog(hs, err_lf, 's-', label='leapfrog')
plt.loglog(hs, err_eu[-1] * (hs / hs[-1]), 'k:', label=r'$\propto h$')           # reference slopes,
plt.loglog(hs, err_lf[-1] * (hs / hs[-1])**2, 'k--', label=r'$\propto h^2$')     # anchored at smallest h
plt.xlabel('step size $h$'); plt.ylabel('position error at $t = 2.3P$'); plt.legend(); plt.show()

ratios = err_lf[:-1] / err_lf[1:]          # error(h) / error(h/2) for each successive pair
print('leapfrog error ratios for successive halvings of h:', np.round(ratios, 3))
print('Euler    error ratios                             :', np.round(err_eu[:-1] / err_eu[1:], 3))
assert np.all((ratios > 3.5) & (ratios < 4.5))

**Figure 2.3.** Position error at $t = 2.3P$ against step size $h$ for Euler (circles) and leapfrog (squares), on log–log axes. Dotted and dashed lines: slopes 1 and 2.

Halving $h$ divides the leapfrog error by 4 (Figure 2.3): it is a **second-order** method. Euler is
first order, but its ratios approach 2 only for the smallest steps: at large $h$ the Euler
orbit is so wrong (Section 2.2) that its error no longer follows a power law. A measured order
is meaningful only in this *convergence regime*, where successive ratios agree.

**Choosing a step size.** For accuracy $\epsilon$ in position, a second-order method such as leapfrog needs a step
$h \propto \epsilon^{1/2}$ ($\propto$: proportional to; for a method of order $p$, $h \propto \epsilon^{1/p}$). Check convergence by halving $h$ and seeing whether your answer changes.

## 2.5 Conservation over many orbits: leapfrog vs a "better" integrator

Higher-order methods are more accurate per step. SciPy's `solve_ivp` uses an adaptive
fifth-order Runge–Kutta method (RK45): it takes trial steps, estimates its own error, and shrinks
or grows the step to keep the error per step below `rtol` (relative) and `atol` (absolute).
It solves $d\mathbf w/dt = f(t, \mathbf w)$ for a single state vector $\mathbf w$, where $f$ is a function we supply that
returns the time derivative of $\mathbf w$; so we pack position and velocity into $\mathbf w = (x, y, v_x, v_y)$, the
Cartesian position and velocity components in the orbital plane.

We compare its energy error with leapfrog over 1000 orbits, for the same number of force
evaluations (the expensive part of any orbit integration).

In [ ]:
def rhs(t, w):
    """Right-hand side for solve_ivp: d/dt (x, v) = (v, g(x)). w[:2] is x, w[2:] is v."""
    return np.concatenate([w[2:], accel_kepler(w[:2])])

n_orbits = 1000
sol = solve_ivp(rhs, (0, n_orbits * P), np.concatenate([x0, v0]), method='RK45',
                rtol=1e-6, atol=1e-9)
# sol.t: times of the integrator's own steps; sol.y: states, shape (4, number of steps)
t_rk = sol.t
dE_rk = np.abs(energy(sol.y[:2].T, sol.y[2:].T, phi_kepler) / E0 - 1)   # .T transposes to (steps, 2)

steps = sol.nfev // n_orbits      # sol.nfev = number of force evaluations RK45 used; // is integer division
x_lf, v_lf = leapfrog(accel_kepler, x0, v0, P / steps, n_orbits * steps)
dE_lf = np.abs(energy(x_lf, v_lf, phi_kepler) / E0 - 1)
t_lf = np.arange(len(x_lf)) * P / steps
print(f'RK45 used {sol.nfev} force evaluations in {len(t_rk)} steps; leapfrog uses {steps} steps per orbit')

# The leapfrog error oscillates within each orbit; plot its maximum per orbit.
# reshape(n_orbits, steps) arranges the errors in one row per orbit; max(axis=1) takes each row's maximum.
dE_lf_max = dE_lf[1:].reshape(n_orbits, steps).max(axis=1)
plt.figure(figsize=(7, 4))
plt.semilogy(np.arange(1, n_orbits + 1), dE_lf_max, label='leapfrog (maximum in each orbit)')
plt.semilogy(t_rk / P, dE_rk, label='RK45 (rtol = 1e-6)')
plt.xlabel('time / orbital period'); plt.ylabel(r'$|\Delta E / E|$'); plt.legend(); plt.show()

first, last = dE_lf[t_lf < 10 * P].max(), dE_lf[t_lf > (n_orbits - 10) * P].max()
rk_10 = dE_rk[t_rk <= 10 * P].max()
print(f'leapfrog max |dE/E|: first 10 orbits {first:.2e}, last 10 orbits {last:.2e}')
print(f'RK45     max |dE/E|: first 10 orbits {rk_10:.2e}, at the end {dE_rk[-1]:.2e}')
# np.argmax on a boolean array returns the index of the first True
print(f'RK45 error first exceeds the leapfrog maximum after {t_rk[np.argmax(dE_rk > last)] / P:.0f} orbits')
assert last < 1.5 * first                 # leapfrog: bounded, no secular drift
assert dE_rk[-1] > 10 * rk_10             # RK45: error keeps growing

**Figure 2.4.** Relative energy error over 1000 orbits for leapfrog (maximum in each orbit) and SciPy's RK45 with `rtol` = 1e-6, at the same number of force evaluations.

For the first few hundred orbits (Figure 2.4) RK45 is *more* accurate — it is a higher-order method. But
its energy error grows steadily, while leapfrog's stays bounded no matter how long we
integrate, so eventually RK45 loses. Stars near the centre of a galaxy complete thousands of
orbits in a Hubble time, and an N-body simulation follows millions of stars: a bounded error
is worth more than a small error per step. This is why most N-body codes use leapfrog.

AGAMA's `agama.orbit` uses an 8th-order Runge–Kutta method, so its energy error also drifts
(Chapter 0 shows this), but at the default `accuracy=1e-8` it is tiny — about $7\times10^{-8}$ per Gyr
for the orbit in Chapter 0 — and it can be reduced further with the `accuracy` argument.

## 2.6 An orbit in the Plummer sphere

Back to the orbit from Chapter 1 §1.3: a star at radius $r = 1$ moving tangentially at 70% of the
circular speed $v_c = \sqrt{GM(r)/r}$ (the speed of a circular orbit at that radius; $M(r)$ is the mass inside $r$). There we *predicted* its pericentre ($0.608888$) and apocentre ($1$) from
the effective potential. Now we integrate it and compare.

In [ ]:
r0 = 1.0
vc0 = np.sqrt(G * M * r0**2 / (r0**2 + b**2)**1.5)          # Plummer circular speed at r0
x0p, v0p = np.array([r0, 0.0]), np.array([0.0, 0.7 * vc0])  # tangential start at 70% of v_c
x_p, v_p = leapfrog(accel_plummer, x0p, v0p, 1e-3, 60000)   # h = 0.001, up to t = 60
r_p = np.linalg.norm(x_p, axis=1)                           # radius at each step
E_p = energy(x_p, v_p, phi_plummer)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(x_p[:, 0], x_p[:, 1], lw=0.6); ax[0].plot(0, 0, 'k*', ms=12)
for rad, ls in [(0.608888, ':'), (1.0, '--')]:              # circles at the predicted r_p and r_a
    ax[0].add_patch(plt.Circle((0, 0), rad, fill=False, ls=ls, color='gray'))
ax[0].set_aspect('equal'); ax[0].set_xlabel('$x$'); ax[0].set_ylabel('$y$')
ax[0].set_title('rosette orbit; circles: predicted $r_p$, $r_a$')
ax[1].plot(np.arange(len(r_p)) * 1e-3, r_p)
ax[1].axhline(0.608888, color='gray', ls=':'); ax[1].axhline(1.0, color='gray', ls='--')
ax[1].set_xlabel('time'); ax[1].set_ylabel('$r$')
plt.tight_layout(); plt.show()

print(f'integrated: r_min = {r_p.min():.6f}, r_max = {r_p.max():.6f};  max |dE/E| = {np.max(np.abs(E_p / E_p[0] - 1)):.1e}')
assert abs(r_p.min() - 0.608888) < 1e-5 and abs(r_p.max() - 1.0) < 1e-5

**Figure 2.5.** Rosette orbit in the Plummer sphere, launched at $r = 1$ with $0.7\,v_c$ and integrated with leapfrog. Left: the orbit in the $x$–$y$ plane, with circles at the predicted pericentre (dotted) and apocentre (dashed). Right: radius against time.

The orbit fills an annulus between the predicted turning points (Figure 2.5). Unlike the Kepler ellipse it
does not close: each radial oscillation the star advances in its azimuthal angle $\phi$ (the polar angle in the
orbital plane, measured from the $x$ axis) by the **apsidal angle** $\Delta\phi$. We measure it from successive apocentres.

`np.arctan2(y, x)` gives the polar angle in $(-\pi, \pi]$, which jumps by $2\pi$ every time the
star crosses the negative $x$-axis; `np.unwrap` removes those jumps so the angle grows continuously.
An apocentre is a step where $r$ is larger than at both neighbouring steps.

In [ ]:
phi = np.unwrap(np.arctan2(x_p[:, 1], x_p[:, 0]))
# r_p[1:-1] is every interior step; r_p[:-2] and r_p[2:] are its previous and next neighbours.
# np.where returns the indices where the condition is True; +1 shifts them back to indices of r_p.
i_apo = np.where((r_p[1:-1] > r_p[:-2]) & (r_p[1:-1] >= r_p[2:]))[0] + 1
dphi = np.diff(phi[i_apo])                 # angle advanced between successive apocentres
print(f'apsidal angle = {dphi.mean():.4f} rad = {np.degrees(dphi.mean()):.1f} deg  (scatter {dphi.std():.1e})')
assert np.pi < dphi.mean() < 2 * np.pi

The apsidal angle lies between $\pi$ (a harmonic potential, where orbits are centred ellipses)
and $2\pi$ (Kepler, where orbits close after one radial period). Chapter 3 explores how it
depends on the orbit.

## 2.7 The same with AGAMA

`agama.orbit` integrates orbits in C++ with an adaptive high-order integrator: by default
`method='dop853'`, an 8th-order Runge–Kutta scheme (Dormand & Prince) that chooses its own step to
keep the error per step below `accuracy` (default $10^{-8}$). We apply the tests of this chapter to
it: accuracy against the exact Kepler orbit, long-term energy conservation, and the Plummer orbit.

First install/import AGAMA (see Chapter 0). As in Chapter 1 §1.5 we do not call `setUnits`, so
AGAMA uses $G = 1$, the units of this chapter.

In [ ]:
import sys, importlib.util, time
if importlib.util.find_spec('agama') is None:
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

pot_kepler = agama.Potential(type='Plummer', mass=M, scaleRadius=0)    # point mass (Chapter 1 §1.5)
pot_plummer = agama.Potential(type='Plummer', mass=M, scaleRadius=b)

### Accuracy and speed on the Kepler orbit

AGAMA always works in 3D, so initial conditions are 6 numbers $(x, y, z, v_x, v_y, v_z)$, the Cartesian position and
velocity; we add zeros for the height $z$ and the vertical velocity $v_z$. `trajsize=2` stores just the start and end points, and `dtype=float` stores
them in double precision (the default, 32-bit floats, would limit us to about 7 digits).
`time.perf_counter()` reads a clock, so the difference of two readings is the elapsed time in seconds.

In [ ]:
ic_kepler = np.concatenate([x0, [0], v0, [0]])      # (x, y, 0, vx, vy, 0)

t0 = time.perf_counter()
_, traj = agama.orbit(potential=pot_kepler, ic=ic_kepler, time=t_end, trajsize=2, dtype=float, verbose=False)
cpu_agama = time.perf_counter() - t0
err_agama = np.linalg.norm(traj[-1, :2] - x_true)  # final position (x, y) vs exact Kepler solution

t0 = time.perf_counter()
nsteps = int(round(t_end / P * 8000))                # our most accurate run in §2.4
x_end = leapfrog(accel_kepler, x0, v0, t_end / nsteps, nsteps)[0][-1]
cpu_lf = time.perf_counter() - t0

print(f'position error at t = 2.3P:  AGAMA {err_agama:.1e} in {cpu_agama * 1e3:.2f} ms;  '
      f'our leapfrog (8000 steps/orbit) {np.linalg.norm(x_end - x_true):.1e} in {cpu_lf * 1e3:.0f} ms')
assert err_agama < 1e-6

AGAMA is hundreds of times more accurate *and* about a hundred times faster. Two reasons: the
8th-order method needs far fewer steps for a given accuracy, and it runs as compiled C++ rather than
a Python loop. (Exact timings depend on your computer.)

### Energy conservation over 1000 orbits

Now repeat §2.5. `trajsize=50001` stores the orbit at 50001 equally spaced times. AGAMA's error
drifts, like RK45's — it is also a Runge–Kutta method — but from a much lower starting level.

In [ ]:
t0 = time.perf_counter()
t_ag, traj = agama.orbit(potential=pot_kepler, ic=ic_kepler, time=n_orbits * P, trajsize=50001, dtype=float, verbose=False)
cpu_1000 = time.perf_counter() - t0
r_ag = np.linalg.norm(traj[:, :3], axis=1)
dE_ag = np.abs((0.5 * np.sum(traj[:, 3:]**2, axis=1) + phi_kepler(r_ag)) / E0 - 1)

plt.figure(figsize=(7, 4))
plt.semilogy(np.arange(1, n_orbits + 1), dE_lf_max, label='our leapfrog (maximum in each orbit)')
plt.semilogy(t_rk / P, dE_rk, label='SciPy RK45 (rtol = 1e-6)')
plt.semilogy(t_ag / P, dE_ag, label='AGAMA (default accuracy 1e-8)')
plt.ylim(1e-9, 1e-1)        # the error is exactly 0 at t = 0, which a log axis cannot show
plt.xlabel('time / orbital period'); plt.ylabel(r'$|\Delta E / E|$'); plt.legend(); plt.show()

print(f'AGAMA: 1000 orbits in {cpu_1000 * 1e3:.0f} ms; |dE/E| after 10 orbits {dE_ag[t_ag <= 10 * P].max():.1e}, '
      f'after 1000 orbits {dE_ag[-1]:.1e}')
assert dE_ag.max() < 1e-4

**Figure 2.6.** Relative energy error over 1000 Kepler orbits for our leapfrog (maximum in each orbit), SciPy's RK45 and `agama.orbit` at its default accuracy.

Over 1000 orbits (Figure 2.6) AGAMA's energy error stays two orders of magnitude below the leapfrog's,
although it grows steadily. The lesson of §2.5 still applies — a non-symplectic method drifts — but
a high-order method with a small tolerance drifts so slowly that it rarely matters for single orbits
in a fixed potential. If it does, reduce `accuracy`.

### The Plummer orbit

Finally, the rosette orbit of §2.6. We store AGAMA's orbit at exactly the times of our leapfrog
steps ($h = 10^{-3}$, 60001 points), so the two can be compared point by point, and check the
turning points against `pot.Rperiapo` (Chapter 1 §1.5).

In [ ]:
ic_plummer = np.concatenate([x0p, [0], v0p, [0]])
t_ag, traj = agama.orbit(potential=pot_plummer, ic=ic_plummer, time=60.0, trajsize=60001, dtype=float, verbose=False)
r_ag = np.linalg.norm(traj[:, :3], axis=1)

E_pl = 0.5 * np.sum(v0p**2) + phi_plummer(r0)        # energy and angular momentum of this orbit
L_pl = r0 * v0p[1]
rp_ag, ra_ag = pot_plummer.Rperiapo([E_pl, L_pl])

dx = np.linalg.norm(x_p - traj[:, :2], axis=1)       # distance between our orbit and AGAMA's

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(t_ag, r_p, label='our leapfrog')
ax[0].plot(t_ag, r_ag, '--', label='AGAMA')
ax[0].axhline(rp_ag, color='gray', ls=':'); ax[0].axhline(ra_ag, color='gray', ls=':')
ax[0].set_xlim(0, 15); ax[0].set_xlabel('time'); ax[0].set_ylabel('$r$'); ax[0].legend()
ax[0].set_title('radius; dotted: Rperiapo')
ax[1].semilogy(t_ag[1:], dx[1:])
ax[1].set_xlabel('time'); ax[1].set_ylabel('|our position - AGAMA position|')
ax[1].set_title('distance between the two orbits')
plt.tight_layout(); plt.show()

print(f'turning points: Rperiapo {rp_ag:.6f}, {ra_ag:.6f};  AGAMA orbit {r_ag.min():.6f}, {r_ag.max():.6f};  '
      f'our orbit {r_p.min():.6f}, {r_p.max():.6f}')
print(f'max distance between the two orbits over t = 0-60: {dx.max():.1e}')
assert abs(r_ag.min() - rp_ag) < 1e-6 and abs(r_ag.max() - ra_ag) < 1e-6
assert dx.max() < 1e-6

# Whose error is it? Compare the final positions at t = 60 for three leapfrog step sizes with a
# very accurate AGAMA orbit (accuracy=1e-12). If the difference is leapfrog's second-order error,
# it must drop by 4 each time h is halved.
_, ref = agama.orbit(potential=pot_plummer, ic=ic_plummer, time=60.0, trajsize=2, dtype=float, accuracy=1e-12, verbose=False)
diffs = []
for h_p in [2e-3, 1e-3, 5e-4]:
    x_end = leapfrog(accel_plummer, x0p, v0p, h_p, int(round(60 / h_p)))[0][-1]
    diffs.append(np.linalg.norm(x_end - ref[-1, :2]))
diffs = np.array(diffs)
print('difference at t = 60 for h = 2e-3, 1e-3, 5e-4:', np.array2string(diffs, precision=2),
      '; ratios', np.round(diffs[:-1] / diffs[1:], 2))
assert np.all(np.abs(diffs[:-1] / diffs[1:] - 4) < 0.5)

**Figure 2.7.** The Plummer orbit of Figure 2.5 integrated by our leapfrog and by AGAMA. Left: radius against time over the first 15 time units; dotted lines: the turning points from `Rperiapo`. Right: distance between the two orbits against time.

The two orbits have the same turning points (Figure 2.7, left) and stay within about $10^{-7}$ of each other (right) over ten
radial periods. The difference is our leapfrog's error, not AGAMA's: halving $h$ divides it by 4,
exactly as a second-order method should (§2.4). From Chapter 3 onwards we use `agama.orbit` for
orbit integration, having checked that it passes our tests.

## Summary

- Euler's method adds energy every step; orbits spiral out.
- Leapfrog (kick–drift–kick) is second order, time-reversible and symplectic: its energy error
  stays bounded over arbitrarily many orbits.
- Check an integrator against an exact solution and measure its order: halve $h$, the error
  should drop by $2^p$.
- Bounded energy error is not the same as accuracy: the phase error still grows.
- A higher-order method is not automatically better for long integrations: RK45's energy drifts.
- Integrated turning points agree with the effective-potential prediction of Chapter 1.
- AGAMA's `agama.orbit` passes the same tests: it is far more accurate and faster than a Python
  leapfrog, and its (non-symplectic) energy drift is small at the default `accuracy`.

## Exercises

1. Write a classical fourth-order Runge–Kutta (RK4) integrator with a fixed step. Measure its
   order with the method of §2.4, then plot its energy error over 1000 orbits as in §2.5.
2. Implement the "drift–kick–drift" variant of leapfrog. Is it still second order?
3. Make the leapfrog step adaptive, $h \propto r^{3/2}$ (smaller near pericentre). What happens
   to the long-term energy error, and why? (Hint: is the scheme still time-reversible?)
4. Integrate the Kepler orbit with $e = 0.9$. How many steps per orbit do you need for a
   position error of $10^{-4}$ after one orbit?
5. Compute the apsidal angle of Plummer orbits with the same apocentre $r = 1$ and
   tangential launch speeds $v$ from 10% to 99% of $v_c$ (here $v$ is the speed at launch, not the vis-viva speed of §2.1).
   Plot $\Delta\phi$ against $v/v_c$. Do it with `agama.orbit`, which accepts an `(N, 6)` array of initial conditions for
   $N$ orbits and integrates all of them in one call.
6. Repeat the AGAMA energy test of §2.7 with `accuracy=1e-6`, `1e-10` and `1e-12`, and with
   `method='dprkn8'` (a Runge–Kutta–Nyström method). How do the error and the run time change?

## References

- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), §3.4.
- J. Bovy, *Dynamics and Astrophysics of Galaxies*, chapter 4: https://galaxiesbook.org
- E. Hairer, C. Lubich & G. Wanner (2003), "Geometric numerical integration illustrated by
  the Störmer–Verlet method", Acta Numerica 12, 399: https://doi.org/10.1017/S0962492902000144